# Absa Uganda app reviews: explore

Uses the functions in `collect_reviews.py`, so the notebook and the hourly job always run the same code. To change how reviews are collected or which columns are added, edit `collect_reviews.py`.

**One-time setup:** `pip install google-play-scraper` (in the same Python that runs this notebook).

## 1. Fetch reviews from each store

In [ ]:
import pandas as pd
from collect_reviews import (DATA_FILE, add_derived_columns, fetch_apple_reviews,
                             fetch_google_reviews, main)

apple_df = fetch_apple_reviews()
google_df = fetch_google_reviews()
print(f"Apple: {len(apple_df)} reviews, Google: {len(google_df)} reviews")
google_df.head()

## 2. Update `data/reviews.csv`

The same step that GitHub runs every hour. Safe to run again: duplicates are removed.

In [ ]:
main()

## 3. Explore the saved data

In [ ]:
df = pd.read_csv(DATA_FILE)
df.groupby("platform").agg(reviews=("review_id", "count"), avg_rating=("rating", "mean"),
                           oldest=("review_day", "min"), newest=("review_day", "max")).round(2)

In [ ]:
# Star split per platform (% of reviews)
(pd.crosstab(df["platform"], df["rating"], normalize="index") * 100).round(1)

In [ ]:
# Reviews and average rating per month (last 12 months)
df.groupby("month").agg(reviews=("review_id", "count"), avg_rating=("rating", "mean")).round(2).tail(12)

In [ ]:
# Newest negative reviews
df[df["sentiment"] == "Negative"].sort_values("review_date_local", ascending=False)[
    ["review_date_local", "platform", "rating", "version", "text"]].head(10)